# Exercise 1: Workflow Observability with Phoenix

Without observability, multi-agent workflows are black boxes. NAT has a **native Phoenix integration** - one YAML block enables full tracing of every LLM call, tool invocation, and agent step.

No manual instrumentation needed. No code changes. Just config.

## Step 1: Launch Phoenix

Phoenix provides a web UI for exploring traces.

In [7]:
import os
import phoenix as px

# In Docker: Phoenix runs as a separate service, no launch needed.
# Locally: launch Phoenix in-process.
phoenix_url = os.getenv("PHOENIX_COLLECTOR_ENDPOINT", "").replace("/v1/traces", "")
if not phoenix_url:
    session = px.launch_app()
    phoenix_url = session.url
    print(f"Phoenix launched: {phoenix_url}")
else:
    print(f"Phoenix service: {phoenix_url}")

print("Open this URL in your browser to view traces.")

Existing running Phoenix instance detected! Shutting it down and starting a new instance...


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Phoenix launched: http://localhost:6006/
Open this URL in your browser to view traces.


In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

True

## Step 2: Review the Config

Compare this to the Part 2 config. The **only difference** is the `general.telemetry` section at the bottom:

In [9]:
with open("config.yaml") as f:
    print(f.read())

# =============================================================
# Exercise 1: Observability with Phoenix Tracing
# =============================================================
# Same workflow as Part 2, with one addition:
# the `general.telemetry` section enables Phoenix tracing
# for every LLM call and tool invocation automatically.
# =============================================================

llms:
  bielik:
    _type: openai
    base_url: "${VLLM_BASE_URL}"
    model: "${VLLM_MODEL_NAME}"
    api_key: "${VLLM_API_KEY}"
    temperature: 0.7
    max_tokens: 2048

functions:
  wiki_search:
    _type: wiki_search
    max_results: 2

  current_time:
    _type: current_datetime

workflow:
  _type: react_agent
  tool_names:
    - wiki_search
    - current_time
  llm_name: bielik
  max_tool_calls: 5
  verbose: true
  parse_agent_response_max_retries: 3
  raise_on_parsing_failure: false
  additional_instructions: "Używaj jak najmniej wywołań narzędzi. Gdy masz wystarczająco informacji, n

That's it. NAT's Phoenix plugin (`nvidia-nat-phoenix`) handles the rest:
- Intercepts every workflow step, tool call, and agent action
- Records inputs, outputs, and timing
- Exports traces to Phoenix via OpenTelemetry

No `OpenAIInstrumentor`, no manual setup. Just YAML.

## Step 3: Run a Workflow with Tracing

In [10]:
!nat run --config_file config.yaml --input "Kim była Maria Skłodowska-Curie i jakie były jej główne osiągnięcia naukowe?"

I0413 20:22:14.268358 6354228 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0413 20:22:14.272606 7175068 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


2026-04-13 20:22:32 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 20:22:34 - INFO     - phoenix.config:2825 - 📋 Ensuring phoenix working directory: /Users/idmochowski/.phoenix
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-04-13 20:22:36 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Num

## Step 4: Explore Traces in Phoenix UI

Open the Phoenix UI and explore:

1. **Traces view** - See all captured workflow runs
2. **Click a trace** - Expand to see workflow spans, tool calls, inputs/outputs
3. **Compare traces** - Which query took the longest? Which used the most tool calls?

Each trace shows:
- `<workflow>` span - the full workflow execution
- `wiki_search` spans - each tool invocation with input query and output content
- Timing for each step

## Step 5: Run More Queries to Build Up Trace Data

In [11]:
queries = [
    "Czym jest Kopalnia Soli w Wieliczce?",
    "Kim był Fryderyk Chopin?",
    "Z czego znana jest Stocznia Gdańska?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print(f"{'='*60}")
    !nat run --config_file config.yaml --input "{query}"


Query 1/3: Czym jest Kopalnia Soli w Wieliczce?
I0413 20:24:52.433945 7191995 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


I0413 20:24:52.428291 6354228 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


2026-04-13 20:24:54 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 20:24:54 - INFO     - phoenix.config:2825 - 📋 Ensuring phoenix working directory: /Users/idmochowski/.phoenix
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-04-13 20:24:54 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Num

I0413 20:24:59.393881 6354228 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0413 20:24:59.397612 7192659 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


2026-04-13 20:25:01 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 20:25:01 - INFO     - phoenix.config:2825 - 📋 Ensuring phoenix working directory: /Users/idmochowski/.phoenix
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-04-13 20:25:01 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Num

I0413 20:25:15.319359 6354228 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


2026-04-13 20:25:17 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 20:25:17 - INFO     - phoenix.config:2825 - 📋 Ensuring phoenix working directory: /Users/idmochowski/.phoenix
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-04-13 20:25:17 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Num

## Step 6: Query Traces Programmatically

In [12]:
from phoenix.client import Client

phoenix_client = Client(base_url=phoenix_url or "http://localhost:6006")
df = phoenix_client.spans.get_spans_dataframe(project_name="bielik-nest-workshop")

print(f"Total spans captured: {len(df)}")
print()

# Show each span
for _, row in df.iterrows():
    name = row.get('name', '?')
    kind = row.get('attributes.openinference.span.kind', '?')
    inp = str(row.get('attributes.input.value', ''))[:60]
    out = str(row.get('attributes.output.value', ''))[:60]
    
    # Calculate duration
    duration = ''
    if 'start_time' in df.columns and 'end_time' in df.columns:
        try:
            dt = (row['end_time'] - row['start_time']).total_seconds() * 1000
            duration = f' ({dt:.0f}ms)'
        except:
            pass
    
    print(f"[{kind}] {name}{duration}")
    print(f"  In:  {inp}...")
    print(f"  Out: {out}...")
    print()

Total spans captured: 26

[CHAIN] wiki_search (3129ms)
  In:  Stocznia Gdańska...
  Out: <Document source="https://en.wikipedia.org/wiki/Gda%C5%84sk_...

[CHAIN] wiki_search (3461ms)
  In:  Stocznia Gdańska...
  Out: <Document source="https://en.wikipedia.org/wiki/Gda%C5%84sk_...

[CHAIN] <workflow> (8099ms)
  In:  Z czego znana jest Stocznia Gdańska?...
  Out: Stocznia Gdańska is known for being the site where the Polis...

[CHAIN] <workflow> (8100ms)
  In:  Z czego znana jest Stocznia Gdańska?...
  Out: Stocznia Gdańska is known for being the site where the Polis...

[CHAIN] wiki_search (2537ms)
  In:  João Gomes (singer)...
  Out: <Document source="https://en.wikipedia.org/wiki/Jo%C3%A3o_Go...

[CHAIN] wiki_search (2986ms)
  In:  Mata (rapper)...
  Out: <Document source="https://en.wikipedia.org/wiki/Mata_(rapper...

[CHAIN] wiki_search (3364ms)
  In:  Kim był Fryderyk Chopin?...
  Out: <Document source="https://en.wikipedia.org/wiki/XVIII_Intern...

[CHAIN] <workflow> (11013ms)
  I

## Bonus: File-Based Tracing (No Phoenix Needed)

NAT also supports writing traces to local files - useful for CI/CD or environments without Phoenix:

```yaml
general:
  telemetry:
    tracing:
      file_traces:
        _type: file
        project: bielik-workshop
        output_path: ./traces
```

Other supported exporters: `langfuse`, `langsmith`, `otelcollector`, `weave`

**Next:** Exercise 2 - Cost tracking with Token Factory pricing model